# 🏠 Modelos predictivos del precio de vivienda — FincaRaíz

Baseline de **valoración automática (AVM)** + enriquecimiento de features.

- **Fuente:** `data/app/housing_clean.parquet` (dataset curado del pipeline).
- **Objetivo:** predecir `Precio` (COP) a partir de área, habitaciones, baños, ubicación y tipo.
- **Contenido:** baseline → `Barrio` (target encoding) → features espaciales → `precio/m²` → **servicios cercanos (POIs OSM)** → conclusiones.

> Requiere `pandas numpy scikit-learn matplotlib pyarrow requests` (todo en `requirements.txt`).

## 1. Cargar datos

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys; sys.path.insert(0, str(ROOT))   # para poder importar src.* desde el notebook
df = pd.read_parquet(ROOT / "data" / "app" / "housing_clean.parquet")
print(f"{len(df):,} inmuebles  ·  {df.shape[1]} columnas")
df.head(3)

## 2. Limpieza / filtros de cordura

`Precio` trae outliers extremos. Filtramos rangos razonables, recortamos precio/m² y exigimos geo válida (la usan las features espaciales).

In [ ]:
n0 = len(df)
df = df[df["Precio"].between(3e7, 5e9)]                        # 30 M – 5.000 M COP
df = df[df["Area_m2"].between(20, 1000)]                        # 20 – 1000 m²
df = df[df["Habitaciones"].between(0, 15) & df["Baños"].between(0, 15)]
df = df.dropna(subset=["Latitud", "Longitud"])                 # geo válida

df["precio_m2"] = df["Precio"] / df["Area_m2"]
lo, hi = df["precio_m2"].quantile([0.01, 0.99])
df = df[df["precio_m2"].between(lo, hi)].reset_index(drop=True)
print(f"{n0:,} → {len(df):,} filas tras el filtrado")

## 3. EDA rápido

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
(df["Precio"]/1e6).plot.hist(bins=60, ax=ax[0], color="#2c7bb6"); ax[0].set_title("Precio (M COP)")
np.log1p(df["Precio"]).plot.hist(bins=60, ax=ax[1], color="#1a9850"); ax[1].set_title("log(1+Precio)")
plt.tight_layout(); plt.show()
print((df.groupby("Tipo_propiedad")["precio_m2"].median()/1e6).round(1).sort_values(ascending=False))

## 4. Features y target (baseline)

Numéricas: `Area_m2, Habitaciones, Baños, Latitud, Longitud`. Categóricas: `Tipo_propiedad, Departamento, Ciudad`. Target: `log(1+Precio)`.

In [ ]:
from sklearn.model_selection import train_test_split
NUM = ["Area_m2", "Habitaciones", "Baños", "Latitud", "Longitud"]
CAT = ["Tipo_propiedad", "Departamento", "Ciudad"]
X = df[NUM + CAT]
y = np.log1p(df["Precio"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)

## 5. Preprocesamiento

Imputación + escala numérica + one-hot categórico (con `min_frequency` para agrupar categorías raras).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False))]), CAT),
])

## 6. Modelos baseline

Evaluación en **pesos reales** (revirtiendo el log): MAE, MAPE mediano, R². `RandomForest` tarda ~1–2 min.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

def metricas(real, pred):
    return {"MAE (M COP)": round(mean_absolute_error(real, pred)/1e6, 1),
            "RMSE (M COP)": round(mean_squared_error(real, pred)**0.5/1e6, 1),
            "MAPE med %": round((np.abs(real-pred)/real).median()*100, 1),
            "R2": round(r2_score(real, pred), 3)}

def evaluar(nombre, modelo, preproc=pre, Xtr=X_train, Xte=X_test, ytr=y_train, yte=y_test):
    pipe = Pipeline([("pre", preproc), ("model", modelo)]).fit(Xtr, ytr)
    m = metricas(np.expm1(yte), np.expm1(pipe.predict(Xte))); m["modelo"] = nombre
    return pipe, m

modelos = {"LinearRegression": LinearRegression(),
           "HistGradientBoost": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08, random_state=42),
           "RandomForest": RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=42)}
pipes, filas = {}, []
for nombre, mdl in modelos.items():
    p, met = evaluar(nombre, mdl); pipes[nombre] = p; filas.append(met); print(met)
resultados = pd.DataFrame(filas).set_index("modelo").sort_values("MAE (M COP)")
resultados

## 7. Importancia de variables

In [ ]:
from sklearn.inspection import permutation_importance
best_name = resultados.index[0]; best = pipes[best_name]
print("Mejor modelo:", best_name)
# n_jobs=1 a propósito: con -1, joblib copia el modelo a /dev/shm (~4 GB en WSL) y la llena.
samp = X_test.sample(min(3000, len(X_test)), random_state=0)
imp = permutation_importance(best, samp, y_test.loc[samp.index], n_repeats=5, random_state=0, n_jobs=1)
(pd.Series(imp.importances_mean, index=NUM + CAT).sort_values()
   .plot.barh(figsize=(7, 4), color="#2c7bb6", title=f"Importancia — {best_name}"))
plt.tight_layout(); plt.show()

## 8. Real vs. predicho

In [ ]:
pred = np.expm1(best.predict(X_test))/1e6; real = np.expm1(y_test).values/1e6
plt.figure(figsize=(5.5, 5.5)); plt.scatter(real, pred, s=6, alpha=0.2, color="#2c7bb6")
lim = [0, np.percentile(real, 99)]; plt.plot(lim, lim, "r--", lw=1); plt.xlim(lim); plt.ylim(lim)
plt.xlabel("Precio real (M COP)"); plt.ylabel("Predicho (M COP)"); plt.title(best_name)
plt.tight_layout(); plt.show()

---
# 🔬 Enriquecimiento de features

A partir de aquí probamos técnicas para mejorar el AVM: **target encoding de `Barrio`**, **features espaciales**,
**target `precio/m²`** y **servicios cercanos (POIs)**. Spoiler honesto: como el modelo ya usa `lat/lon`, varias
aportan poco; al final resumimos qué **sí** mueve la aguja.

## 9. Feature engineering espacial (offline)

- **`dist_centro_km`**: distancia (haversine) al centro de su ciudad (mediana de lat/lon).
- **`densidad_1km`**: nº de avisos a ≤1 km (proxy de zona activa).
- **`geo_cluster`**: zona geográfica por KMeans sobre lat/lon.
- **`freq_barrio`**: nº de avisos del barrio (frequency encoding).

In [ ]:
from sklearn.neighbors import BallTree
from sklearn.cluster import KMeans

def haversine(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = map(np.radians, [la1, lo1, la2, lo2])
    d = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(d))

cen = df.groupby("Ciudad")[["Latitud", "Longitud"]].transform("median")
df["dist_centro_km"] = haversine(df.Latitud, df.Longitud, cen.Latitud, cen.Longitud)

coords = np.radians(df[["Latitud", "Longitud"]].values)
tree = BallTree(coords, metric="haversine")
df["densidad_1km"] = tree.query_radius(coords, r=1/6371, count_only=True) - 1     # -1: no contarse a sí mismo
df["geo_cluster"] = KMeans(n_clusters=40, n_init=5, random_state=0).fit_predict(df[["Latitud", "Longitud"]]).astype(str)
df["freq_barrio"] = df.groupby("Barrio")["Barrio"].transform("count")
df[["dist_centro_km", "densidad_1km", "geo_cluster", "freq_barrio"]].describe().round(2)

## 10. `Barrio` con target encoding + features espaciales

`Barrio` es de alta cardinalidad. Usamos **`TargetEncoder` de scikit-learn** (hace cross-fitting interno, evita fuga
de información — equivalente a `category_encoders` pero sin dependencia extra). Comparamos baseline vs enriquecido.

In [ ]:
from sklearn.preprocessing import TargetEncoder

def construir_pre(num, oh, te):
    t = [("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
         ("oh", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="NA")),
                          ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False))]), oh)]
    if te: t.append(("te", TargetEncoder(target_type="continuous"), te))
    return ColumnTransformer(t)

def comparar(nombre, num, oh, te, target="Precio"):
    X = df[list(dict.fromkeys(num + oh + te))]
    y = np.log1p(df[target])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    m = Pipeline([("pre", construir_pre(num, oh, te)),
                  ("m", HistGradientBoostingRegressor(max_iter=500, learning_rate=0.08, random_state=42))]).fit(Xtr, ytr)
    if target == "Precio":
        pred, real = np.expm1(m.predict(Xte)), np.expm1(yte)
    else:  # precio_m2 → reconstruir precio
        area = df.loc[Xte.index, "Area_m2"]; pred = np.expm1(m.predict(Xte)) * area; real = df.loc[Xte.index, "Precio"]
    out = metricas(real, pred); out["modelo"] = nombre; return out

NUM_EXT = NUM + ["dist_centro_km", "densidad_1km", "freq_barrio"]
OH = ["Tipo_propiedad", "Departamento"]; TE = ["Ciudad", "Barrio", "geo_cluster"]

comp = pd.DataFrame([
    comparar("baseline",                    NUM,     OH, ["Ciudad"]),
    comparar("+ Barrio (target enc.)",      NUM,     OH, ["Ciudad", "Barrio"]),
    comparar("+ espaciales + clusters",     NUM_EXT, OH, TE),
]).set_index("modelo")
comp

> **Lectura honesta:** las mejoras suelen ser pequeñas (~+0.005 de R²). Con `lat/lon` preciso ya en el modelo,
`Barrio`, clusters y densidad son en buena parte **redundantes**. Target encoding brilla más en modelos lineales
o cuando no tienes coordenadas.

## 11. Target alternativo: `precio/m²`

Modelar `log(precio/m²)` y reconstruir `Precio = precio_m2 × Area` suele ser **más estable** entre zonas y tamaños.

In [ ]:
pd.DataFrame([
    comparar("target = Precio",     NUM_EXT, OH, TE, target="Precio"),
    comparar("target = precio_m2",  NUM_EXT, OH, TE, target="precio_m2"),
]).set_index("modelo")

## 12. Modelo por ciudad (dónde se predice mejor)

El R² varía mucho por ciudad. Entrenamos un modelo por ciudad (con suficientes datos) y comparamos.

In [ ]:
def r2_ciudad(ciudad, min_n=800):
    d = df[df.Ciudad == ciudad]
    if len(d) < min_n: return None
    X = d[NUM_EXT + ["Tipo_propiedad", "Barrio", "geo_cluster"]]; y = np.log1p(d["Precio"])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    pre_c = construir_pre(NUM_EXT, ["Tipo_propiedad"], ["Barrio", "geo_cluster"])
    m = Pipeline([("pre", pre_c), ("m", HistGradientBoostingRegressor(max_iter=400, random_state=42))]).fit(Xtr, ytr)
    return round(r2_score(np.expm1(yte), np.expm1(m.predict(Xte))), 3)

top = df["Ciudad"].value_counts().head(8).index
por_ciudad = {c: r2_ciudad(c) for c in top}
pd.Series({k: v for k, v in por_ciudad.items() if v is not None}, name="R2").sort_values(ascending=False)

## 13. Servicios cercanos (POIs) — extract OSM offline (nacional)

El **contexto urbano** (supermercados, colegios, salud, parques, transporte) puede influir en el precio. Para calcularlo
a escala nacional **sin costo ni límites de API**, usamos un *extract* de Colombia de OpenStreetMap:

| Fuente | Costo | Escala |
|---|---|---|
| **OSM extract** (`.pbf` Geofabrik) + DuckDB | **Gratis, offline** | ✅ 100k sin límites |
| Overpass (API en vivo) | Gratis | Bueno para zonas/demos |
| Google Places API | ~$17/1000 | Inviable a 100k por costo |

`src/osm_pois.py` extrae los POIs del `.pbf` con **DuckDB** (`ST_ReadOSM`) → `data/osm/pois_colombia.parquet`, y
`features_servicios(df)` añade: nº de servicios por radio, nº por categoría a 1 km y **distancia a la más cercana**.

> Setup (una vez): baja el `.pbf` de [Geofabrik](https://download.geofabrik.de/south-america/colombia.html) a `data/osm/`,
> `pip install duckdb` y `python -m src.osm_pois`. (`pyrosm` no compila en Python 3.12 → usamos DuckDB.)
> El `pois_colombia.parquet` (1.5 MB) puede commitearse para correr el notebook sin el `.pbf`.

In [ ]:
# Servicios cercanos desde el extract OSM (usa data/osm/pois_colombia.parquet)
try:
    from src.osm_pois import features_servicios
    dfx = features_servicios(df)                        # + serv_*, n_*_1km, dist_*_km
    osm_cols = [c for c in dfx.columns if c.startswith(("serv_", "n_", "dist_"))]
    print(f"{len(osm_cols)} features de servicios:", osm_cols[:6], "…")

    def r2_serv(cols):
        X = dfx[cols + ["Tipo_propiedad", "Departamento", "Ciudad"]]; y = np.log1p(dfx["Precio"])
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
        m = Pipeline([("pre", construir_pre(cols, ["Tipo_propiedad", "Departamento"], ["Ciudad"])),
                      ("m", HistGradientBoostingRegressor(max_iter=500, random_state=42))]).fit(Xtr, ytr)
        return round(r2_score(np.expm1(yte), np.expm1(m.predict(Xte))), 3)

    print("R2 sin servicios:", r2_serv(NUM))
    print("R2 con servicios:", r2_serv(NUM + osm_cols))
    print(dfx[["Ciudad"] + osm_cols[:5]].head().to_string())
except FileNotFoundError:
    print("Falta data/osm/pois_colombia.parquet.\n"
          "Baja el .pbf de Geofabrik a data/osm/, luego: pip install duckdb && python -m src.osm_pois")

> **Lectura honesta (medido a nivel nacional):** R² 0.803 → **0.807 (+0.003)**. Con `lat/lon` + `Ciudad` ya en el
modelo, los servicios son en buena parte **redundantes para *predecir***. Su valor real: (1) **interpretabilidad /
segmentación** ('4 colegios y 2 supermercados a 1 km'), (2) modelos **sin coordenadas**, (3) señales **específicas**
(distancia al metro/estación). Aun con lift chico, deja el *feature store* de servicios listo para esos usos.

## 14. Optimización: grilla de micro-zonas + caché (POIs baratos)

Consultar POIs **por inmueble** es carísimo (100k llamadas). Truco: **redondear coordenadas** a una grilla y hacer
**1 llamada por micro-zona**; todos los avisos de esa celda comparten el resultado. Medido con estos datos:

| Redondeo | Celda | Celdas únicas | Reducción |
|---|---|---|---|
| **2 dec** | **~1.1 km** | **4.319** | **25.9×** ✅ cabe en la capa gratis |
| 3 dec | ~110 m | 37.323 | 3.0× |

Con **2 decimales**, las ~112k viviendas se resuelven con **~4.300 llamadas** (bajo el free tier de ~5.000 de Google).
Y con **caché persistente**, es costo **de una sola vez**: los avisos nuevos caen casi siempre en celdas ya resueltas
→ llamadas ≈ 0 después. La misma función sirve para **Overpass (gratis)** o **Google Places** (basta cambiar el proveedor).

In [ ]:
import os, requests
GRID_DEC = 2   # 2 decimales ≈ celdas de ~1.1 km → 1 llamada por micro-zona

df["celda"] = df.Latitud.round(GRID_DEC).astype(str) + "," + df.Longitud.round(GRID_DEC).astype(str)
centros = df.groupby("celda")[["Latitud", "Longitud"]].mean()
print(f"{len(df):,} inmuebles → {len(centros):,} micro-zonas ({len(df)/len(centros):.1f}x menos llamadas)")

CACHE = ROOT / "data" / "poi_cache.parquet"
cache = pd.read_parquet(CACHE) if CACHE.exists() else pd.DataFrame(columns=["servicios"], index=pd.Index([], name="celda"))

def servicios_overpass(lat, lon, radio_m=800):
    q = (f'[out:json][timeout:30];('
         f'nwr(around:{radio_m},{lat},{lon})["amenity"~"school|hospital|pharmacy|bank|restaurant"];'
         f'nwr(around:{radio_m},{lat},{lon})["shop"="supermarket"];);out count;')
    r = requests.post("https://overpass-api.de/api/interpreter", data={"data": q},
                      headers={"User-Agent": "fr_scraper-nb"}, timeout=40)
    return int(r.json()["elements"][0]["tags"]["total"])

# (opcional) Google Places — gracias a la grilla cabe en la capa gratis:
# def servicios_google(lat, lon, radio_m=800):
#     r = requests.get("https://maps.googleapis.com/maps/api/place/nearbysearch/json",
#                      params={"location": f"{lat},{lon}", "radius": radio_m,
#                              "type": "point_of_interest", "key": os.getenv("GOOGLE_MAPS_API_KEY")})
#     return len(r.json().get("results", []))

# Rellena SOLO celdas nuevas (demo: 25 para no tardar; en real, todas una vez y queda cacheado)
nuevas = [c for c in centros.index if c not in cache.index][:25]
filas = []
for c in nuevas:
    lat, lon = centros.loc[c]
    try: filas.append({"celda": c, "servicios": servicios_overpass(lat, lon)})
    except Exception: pass
if filas:
    cache = pd.concat([cache, pd.DataFrame(filas).set_index("celda")])
    cache.to_parquet(CACHE)
print(f"caché: {len(cache):,} celdas resueltas (persistida en data/{CACHE.name})")

df = df.join(cache, on="celda")   # cada aviso hereda 'servicios' de su micro-zona
df[["Ciudad", "celda", "servicios"]].dropna().head()

## 15. Conclusiones — qué mueve la aguja

- **`Area_m2` domina** por lejos; `lat/lon` captura casi toda la señal de ubicación.
- **Barrio target-enc, clusters, densidad y POIs agregados → lift marginal** (~+0.005 R²) porque son redundantes con `lat/lon`.
- **`precio/m²` como target** suele dar predicciones más estables entre segmentos.
- **Modelo por ciudad**: el R² varía mucho; en mercados grandes conviene un modelo dedicado.

**Dónde está el verdadero lift (próximos pasos):**
- **Atributos que hoy NO tenemos**: estrato, parqueaderos, antigüedad, piso, amenidades → habría que scrapear el detalle.
- **Visión por computador** sobre las fotos (estado/calidad).
- **POIs específicos** (distancia al metro/estaciones, colegios de calidad) vía OSM offline (extract de Geofabrik).
- **Intervalos de predicción** (quantile / conformal) para dar un rango.
- **Detección de oportunidades**: inmuebles con `precio_real ≪ predicho` (residual negativo grande) = posibles gangas.
- **XGBoost/LightGBM** + `RandomizedSearchCV` para exprimir el último punto de R².